# ACDL Sparse MoE: real-evidence walkthrough

This notebook audits completed development-run artifacts and provides an opt-in shared-prefix inference run. Before execution, set `ACDL_DATA_DIR` to the prepared development dataset and point `ACDL_RUN_ROOT` at a completed E0-E3 evidence root. The fallback `outputs/final_selected` is an intentionally unresolved placeholder. Set `ACDL_RUN_INFERENCE=1` only when the required checkpoint, data, and runtime dependencies are available.


In [ ]:
import json
import os
import subprocess
from pathlib import Path



def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "run.sh").is_file():
            return candidate
    raise RuntimeError("Run this notebook from the sparse_moe repository.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if "ACDL_DATA_DIR" not in os.environ:
    raise RuntimeError("Set ACDL_DATA_DIR to the development-data directory.")

run_root_setting = os.environ.get("ACDL_RUN_ROOT", "outputs/final_selected")
data_dir_setting = os.environ["ACDL_DATA_DIR"]


def repo_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (REPO_ROOT / path).resolve()


RUN_ROOT = repo_path(run_root_setting)
DATA_DIR = repo_path(data_dir_setting)
RUN_MANIFEST_PATH = RUN_ROOT / "manifest.json"
DATA_MANIFEST_PATH = DATA_DIR / "manifest.json"
E3_METRICS_PATH = RUN_ROOT / "E3_final_multimodal_top2" / "metrics.json"
E3_TEXT_METADATA_PATH = RUN_ROOT / "E3_final_multimodal_top2_text_eval" / "metrics.json"
CHECKPOINT_PATH = RUN_ROOT / "E3_final_multimodal_top2" / "checkpoint_final.pt"


def load_json(path):
    if not path.is_file():
        raise FileNotFoundError(f"Missing required JSON artifact: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def markdown_table(headers, rows):
    result = [
        "| " + " | ".join(map(str, headers)) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    result.extend("| " + " | ".join(map(str, row)) + " |" for row in rows)
    return chr(10).join(result)


run_manifest = load_json(RUN_MANIFEST_PATH)
data_manifest = load_json(DATA_MANIFEST_PATH)
e3_metrics = load_json(E3_METRICS_PATH)
e3_text_metadata = load_json(E3_TEXT_METADATA_PATH)
checkpoint_provenance = e3_text_metadata["provenance"]

assert e3_metrics.get("real_subset") is True
assert e3_text_metadata.get("real_subset") is True
assert checkpoint_provenance["source_experiment_id"] == "E3_final_multimodal_top2"

provenance = [
    ("ACDL_RUN_ROOT", run_root_setting),
    ("resolved run root", RUN_ROOT),
    ("ACDL_DATA_DIR", data_dir_setting),
    ("resolved data directory", DATA_DIR),
    ("Run:AI job", run_manifest.get("runai_job_name", "not recorded")),
    ("base model", run_manifest["base_model"]),
    ("vision model", run_manifest["vision_model"]),
    ("speech model", run_manifest["speech_model"]),
    ("checkpoint metadata", E3_TEXT_METADATA_PATH),
    ("E3 checkpoint", CHECKPOINT_PATH),
    ("checkpoint source experiment", checkpoint_provenance["source_experiment_id"]),
]
for key, value in provenance:
    print(f"{key}: {value}")

## Dataset manifest checks

The external development manifest must match the copy embedded in the run manifest. The split declarations are checked where the run records a corresponding train/eval split.

In [ ]:
run_data_manifest = run_manifest["data_manifest"]
assert data_manifest.get("errors") == []
assert data_manifest["counts"] == run_data_manifest["counts"]
assert data_manifest["sources"] == run_data_manifest["sources"]

count_specs = [
    ("text tasks", "text_tasks", None),
    ("text train blocks", "text_blocks_train", "text_train_blocks"),
    ("text eval blocks", "text_blocks_eval", "text_eval_blocks"),
    ("image captions", "image_captions", None),
    ("image train pairs", "image_train_pairs", "image_train_pairs"),
    ("image eval pairs", "image_eval_pairs", "image_eval_pairs"),
    ("speech transcripts", "speech_transcripts", None),
    ("speech train utterances", "speech_train_utterances", "speech_train_utterances"),
    ("speech eval utterances", "speech_eval_utterances", "speech_eval_utterances"),
]
count_rows = []
for label, count_key, split_key in count_specs:
    data_count = data_manifest["counts"][count_key]
    embedded_count = run_data_manifest["counts"][count_key]
    split_count = run_manifest["splits"].get(split_key) if split_key else "-"
    assert data_count == embedded_count
    if split_key:
        assert data_count == split_count
    count_rows.append((label, data_count, embedded_count, split_count))

print(markdown_table(
    ("dataset item", "data manifest", "run copy", "run split"),
    count_rows,
))
print()
print("Dataset sources:")
for modality, source in sorted(data_manifest["sources"].items()):
    print(f"- {modality}: {source}")

## E0-E3 evidence table

The first three rows use their experiment metrics directly. The final row uses the E3-owned text-evaluation object embedded in E3 metrics, preserving its checkpoint provenance.

In [ ]:
EXPERIMENT_FILES = {
    "Top-8": "E0_top8_teacher_baseline/metrics.json",
    "hard-Top2": "E1_hard_top2/metrics.json",
    "calibrated": "E2_calibrated_top2/metrics.json",
    "final": "E3_final_multimodal_top2/metrics.json",
}
experiment_metrics = {
    label: load_json(RUN_ROOT / relative_path)
    for label, relative_path in EXPERIMENT_FILES.items()
}
assert all(metrics.get("real_subset") is True for metrics in experiment_metrics.values())

evaluation_metrics = {
    "Top-8": experiment_metrics["Top-8"],
    "hard-Top2": experiment_metrics["hard-Top2"],
    "calibrated": experiment_metrics["calibrated"],
    "final": experiment_metrics["final"]["text_eval"],
}
assert evaluation_metrics["final"]["provenance"]["copied_from_e2"] is False


def fmt_number(value):
    return f"{float(value):.3f}"


def fmt_percent(value):
    return f"{100.0 * float(value):.2f}%"


metric_rows = []
for label, metrics in evaluation_metrics.items():
    metric_rows.append((
        label,
        metrics["runtime_top_k"],
        fmt_number(metrics["perplexity"]),
        fmt_percent(metrics["next_token_accuracy"]),
        fmt_number(metrics["gate_entropy_mean"]),
        fmt_percent(metrics["inactive_expert_ratio_mean"]),
        fmt_percent(metrics["capacity_overflow_ratio_mean"]),
    ))

print(markdown_table(
    ("experiment", "top-k", "text PPL", "token acc.", "gate entropy", "inactive", "dropped"),
    metric_rows,
))

## E3 checkpoint metadata

This check inspects JSON metadata and filesystem metadata only. It does not deserialize checkpoint tensors.

In [ ]:
expected_metadata_path = RUN_ROOT / "E3_final_multimodal_top2_text_eval" / "metrics.json"
assert E3_TEXT_METADATA_PATH == expected_metadata_path

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Missing E3 checkpoint: {CHECKPOINT_PATH}")

recorded_checkpoint = Path(e3_metrics["checkpoint_path"])
provenance_checkpoint = Path(checkpoint_provenance["source_checkpoint"])
checkpoint_stat = CHECKPOINT_PATH.stat()
expected_suffix = CHECKPOINT_PATH.parts[-2:]
assert recorded_checkpoint.parts[-2:] == expected_suffix
assert provenance_checkpoint.parts[-2:] == expected_suffix
assert e3_metrics["checkpoint_size_bytes"] == checkpoint_stat.st_size
assert checkpoint_provenance["source_checkpoint_size_bytes"] == checkpoint_stat.st_size
assert checkpoint_provenance["source_checkpoint_saved_before_eval"] is True

checkpoint_rows = [
    ("E3 metrics path", E3_METRICS_PATH),
    ("checkpoint path in E3 metadata", e3_metrics["checkpoint_path"]),
    ("text-eval metadata path", E3_TEXT_METADATA_PATH),
    ("source experiment", checkpoint_provenance["source_experiment_id"]),
    ("training steps", checkpoint_provenance["source_training_steps"]),
    ("checkpoint bytes", checkpoint_stat.st_size),
    ("saved before text eval", checkpoint_provenance["source_checkpoint_saved_before_eval"]),
    ("copied from E2", checkpoint_provenance["copied_from_e2"]),
]
print(markdown_table(("field", "value"), checkpoint_rows))

## Text and routing metrics

All plotted values come from the JSON objects used in the table above. Overflow is the capacity-dropped routing fraction; inactive is the fraction of experts with no routed tokens.

In [ ]:
from html import escape

labels = list(evaluation_metrics)
plot_panels = [
    (
        "Text PPL",
        True,
        [("PPL", [evaluation_metrics[label]["perplexity"] for label in labels], "#2563eb")],
    ),
    (
        "Routing entropy",
        False,
        [(
            "gate entropy",
            [evaluation_metrics[label]["gate_entropy_mean"] for label in labels],
            "#15803d",
        )],
    ),
    (
        "Routing/drop (%)",
        True,
        [
            (
                "capacity dropped",
                [
                    100.0 * evaluation_metrics[label]["capacity_overflow_ratio_mean"]
                    for label in labels
                ],
                "#dc2626",
            ),
            (
                "inactive experts",
                [
                    100.0 * evaluation_metrics[label]["inactive_expert_ratio_mean"]
                    for label in labels
                ],
                "#7c3aed",
            ),
        ],
    ),
]


class InlineSVG:
    def __init__(self, content):
        self.content = content

    def _repr_svg_(self):
        return self.content


def tick(value):
    return f"{value:.2f}" if abs(value) < 10 else f"{value:.1f}"


svg = [
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 320" '
    'role="img" aria-label="Text perplexity and routing metrics">',
    '<rect width="960" height="320" fill="white"/>',
]
for panel_index, (title, zero_floor, series) in enumerate(plot_panels):
    panel_x = 10 + 315 * panel_index
    plot_x, plot_y, plot_width, plot_height = panel_x + 35, 54, 240, 180
    values = [value for _, points, _ in series for value in points]
    lower = 0.0 if zero_floor else min(values) * 0.98
    upper = max(values) * 1.08
    if upper <= lower:
        upper = lower + 1.0

    svg.extend([
        f'<text x="{plot_x + plot_width / 2}" y="24" text-anchor="middle" '
        f'font-family="sans-serif" font-size="15" font-weight="600">{escape(title)}</text>',
        f'<line x1="{plot_x}" y1="{plot_y}" x2="{plot_x}" y2="{plot_y + plot_height}" '
        'stroke="#64748b"/>',
        f'<line x1="{plot_x}" y1="{plot_y + plot_height}" '
        f'x2="{plot_x + plot_width}" y2="{plot_y + plot_height}" stroke="#64748b"/>',
        f'<text x="{plot_x - 5}" y="{plot_y + 4}" text-anchor="end" '
        f'font-family="sans-serif" font-size="10">{tick(upper)}</text>',
        f'<text x="{plot_x - 5}" y="{plot_y + plot_height + 4}" text-anchor="end" '
        f'font-family="sans-serif" font-size="10">{tick(lower)}</text>',
    ])

    x_values = [
        plot_x + index * plot_width / (len(labels) - 1)
        for index in range(len(labels))
    ]
    for x_value, label in zip(x_values, labels):
        svg.append(
            f'<text x="{x_value}" y="{plot_y + plot_height + 18}" text-anchor="middle" '
            f'font-family="sans-serif" font-size="10">{escape(label)}</text>'
        )

    for series_index, (name, points, color) in enumerate(series):
        y_values = [
            plot_y + plot_height * (upper - value) / (upper - lower)
            for value in points
        ]
        coordinates = " ".join(
            f"{x_value:.1f},{y_value:.1f}"
            for x_value, y_value in zip(x_values, y_values)
        )
        svg.append(
            f'<polyline points="{coordinates}" fill="none" stroke="{color}" '
            'stroke-width="2.5"/>'
        )
        svg.extend(
            f'<circle cx="{x_value:.1f}" cy="{y_value:.1f}" r="3.5" fill="{color}"/>'
            for x_value, y_value in zip(x_values, y_values)
        )
        legend_x = plot_x + series_index * 125
        svg.extend([
            f'<line x1="{legend_x}" y1="292" x2="{legend_x + 18}" y2="292" '
            f'stroke="{color}" stroke-width="2.5"/>',
            f'<text x="{legend_x + 23}" y="296" font-family="sans-serif" '
            f'font-size="10">{escape(name)}</text>',
        ])

svg.append("</svg>")
InlineSVG(chr(10).join(svg))


## Development-only shared-prefix conditional evaluation

The next cell builds the exact command and executes it only when `ACDL_RUN_INFERENCE=1`; otherwise it prints the command. Keep `ACDL_DATA_DIR` pointed at development data and choose a fresh `ACDL_DEV_EVAL_DIR`. The checkpoint is read-only, while diagnostics are written outside the selected run root.

`metrics.json` contains aggregate image/speech conditional R@1, chance rates, query/candidate counts, and evaluation-path provenance. `per_query.jsonl` contains query-level candidate rankings and conditional NLL values for auditing failures.


In [ ]:
conditional_eval_command = r'''DEV_EVAL_DIR="${ACDL_DEV_EVAL_DIR:-outputs/conditional_eval_dev}"
mkdir -p "$DEV_EVAL_DIR"
MODE=conditional-eval \
DATA_DIR="$ACDL_DATA_DIR" \
RUN_OUTPUT_DIR="$ACDL_RUN_ROOT" \
CHECKPOINT="$ACDL_RUN_ROOT/E3_final_multimodal_top2/checkpoint_final.pt" \
OUT="$DEV_EVAL_DIR/metrics.json" \
PER_QUERY_OUTPUT="$DEV_EVAL_DIR/per_query.jsonl" \
EVAL_PATH=shared_prefix \
PREFIX_CONTROL=real \
NEGATIVE_MODE=stride \
CONDITIONAL_QUERIES=50 \
CONDITIONAL_CANDIDATES=10 \
CONDITIONAL_NEGATIVES=9 \
bash run.sh conditional-eval'''
run_inference = os.environ.get("ACDL_RUN_INFERENCE", "0") == "1"
if run_inference:
    subprocess.run(
        ["bash", "-lc", conditional_eval_command],
        cwd=REPO_ROOT,
        env=os.environ.copy(),
        check=True,
    )
else:
    print(conditional_eval_command)
